# Analyse des déterminants de l'équipement en Véhicules Électriques

## 1. Introduction

Ce projet vise à comprendre quels facteurs (socio-économiques, infrastructures existantes) influencent le taux d'équipement en véhicules électriques au niveau communal en France.

Nous fusionnons trois sources de données provenant de data.gouv.fr :
- IRVE : Données sur les infrastructures de recharge.
- INSEE : Revenus et données socio-économiques.
- Immatriculations : Parc de VE par commune.

## 2. Chargement et Exploration des Données Brutes

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.loading import load_all_datasets, charger_communes
from src.cleaning import standardize_all_codes, clean_irve_variables_finales, corriger_codes_incoherents, corriger_par_nom, corriger_conflit_code_postal, garder_derniere_observation_commune
from src.features import preparer_base_ve, creer_features_irve, creer_taux_equipement_ve
from src.modelisation import modelisation_lasso, modelisation_xgboost
from src.utils import diagnostic_cle_jointure, creer_gdf_irve, joindre_communes, ajouter_codes_geo, compter_valeurs_manquantes, compter_uniques

# Configuration des chemins
PATHS = {
    'irve': 'https://www.data.gouv.fr/api/1/datasets/r/eb76d20a-8501-400e-b336-d85724de5435',
    'revenu': 'https://www.data.gouv.fr/api/1/datasets/r/516130bc-4dcb-47f5-8347-ae96553c43ab',
    've': 'https://www.data.gouv.fr/api/1/datasets/r/90e0d717-deda-4bdc-9987-f82faac5bc93',
}

# Chargement
df_irve_raw, df_revenu_raw, df_ve_raw = load_all_datasets(PATHS)

#### IRVE

In [2]:
print(f"Shape : {df_irve_raw.shape}")
df_irve_raw.sample(5)

Shape : (217227, 52)


,nom_amenageur,siren_amenageur,contact_amenageur,nom_operateur,contact_operateur,telephone_operateur,nom_enseigne,id_station_itinerance,id_station_local,nom_station,...,datagouv_resource_id,datagouv_organization_or_owner,created_at,consolidated_longitude,consolidated_latitude,consolidated_code_postal,consolidated_commune,consolidated_is_lon_lat_correct,consolidated_is_code_insee_verified,consolidated_is_code_insee_modified
38381,442__Easycharge,NaN,NaN,Easy Charge | FR*ECH,roaming@freshmile.com,NaN,Easy Charge,FRECHP5248546983339760491,1396582,Easy Charge/f737f926-b457-5e97-8e1c-abddabe636d7,...,61387a4e-22f7-4662-b241-d5cac4dd91fd,gireve-2,2023-03-24T14:32:54.036000+00:00,6.081773,44.562128,NaN,NaN,False,False,False
51320,E-TOTEM,NaN,contact@e-totem.fr,E-TOTEM,contact@e-totem.fr,NaN,Réseau e-Totem Infrastructures,FRETIP44143A,FRETIP44143A,e-Totem - NANTES - Rezé - Gymnase Arthur Dugast,...,53f9d331-b845-453c-977d-cd6c48e5bb52,e-totem,2025-06-17T11:55:33.909000+00:00,-1.545033,47.165286,44400.0,Rezé,True,True,False
71099,Syndicat d'Energie de l'Oise - SE60,NaN,NaN,Bouygues Energies & Services,support@alizecharge.fr,0805021480,Pass pass électrique,FRH01E60703001,NaN,AUX MARAIS - Route De Gisors,...,5f29737c-5393-46f9-8140-2509992adc7a,alize,2025-03-17T08:58:10.352000+00:00,2.043551,49.413887,60000.0,Aux Marais,True,True,False
125751,Power Dot France,891118473.0,hello@powerdot.fr,Power Dot France,hello@powerdot.fr,tel:+33-1-76-31-06-84,Power Dot France,FRPD1PINTECR,NaN,Intermarché - Écrouves,...,8bb0a6e2-1016-42ba-aaee-f72f55c82e9f,qualicharge,2025-05-05T13:28:02.361000+00:00,5.872424,48.679686,54200.0,Écrouves,True,True,False
163817,SDEM50,255002883.0,sdem@sdem50.fr,Zunder (Grupo Easycharger S.A),e-charge50@sdem50.fr,0187650079,e-charge50,FRS50P50147006,NaN,COUTANCES - Cathédrale,...,25c7d722-c9fd-475a-95fb-f5697893e3a9,e-charge50,2026-03-16T14:32:29.148000+00:00,-1.446090,49.047760,50200.0,Coutances,True,True,False


Ce premier jeu de données est la base nationale des IRVE (Infrastructures de Recharge pour Véhicules Électriques).
Cette table comporte 212234 lignes pour 52 variables.

#### Véhicules

In [3]:
print(f"Shape : {df_ve_raw.shape}")
df_ve_raw.sample(5)

Shape : (703545, 8)


,CODGEO,LIBGEO,EPCI,LIBEPCI,DATE_ARRETE,NB_VP_RECHARGEABLES_EL,NB_VP_RECHARGEABLES_GAZ,NB_VP
467916,30229,SAINT-ANDRÉ-DE-MAJENCOULES,200034601,CC Causses Aigoual Cévennes,2024-06-30,12,0,784
256470,34049,CAMPLONG,200042646,CC Grand Orb,2021-09-30,0,0,265
96772,79157,LOUZY,247900798,CC du Thouarsais,2025-09-30,35,0,1337
295238,65221,HIIS,246500482,CC de la Haute-Bigorre,2024-06-30,6,0,286
317701,85153,MOUCHAMPS,248500621,CC du Pays des Herbiers,2022-09-30,11,0,2973


Ce second jeu de données contient des informations sur les voitures particulières immatriculées par commune et par type de recharge. Il contient l'historique des immatriculations pour chaque commune à différentes dates.
Cette table comporte 703545 lignes pour 8 variables. 

#### Revenus

In [4]:
print(f"Shape : {df_revenu_raw.shape}")
df_revenu_raw.sample(5)

Shape : (34926, 57)


,Nom géographique GMS,Code géographique,Libellé géographique,[DISP] Nbre de ménages fiscaux,[DISP] Nbre de personnes dans les ménages fiscaux,[DISP] Nbre d'unités de consommation dans les ménages fiscaux,[DISP] 1ᵉʳ quartile (€),[DISP] Médiane (€),[DISP] 3ᵉ quartile (€),[DISP] Écart interquartile (€),...,[DEC] 9ᵉ décile (€),[DEC] Rapport interdécile D9/D1,[DEC] S80/S20,[DEC] Iice de Gini,[DEC] Part des revenus d’activité (%),[DEC] dont part des salaires et traitements (%),[DEC] dont part des iemnités de chômage (%),[DEC] dont part des revenus des activités non salariées (%),"[DEC] Part des pensions, retraites et rentes (%)",[DEC] Part des autres revenus (%)
31360,Franleu,80345,Franleu,209.0,480.0,328.2,NaN,21390.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9858,Thiberville,27629,Thiberville,830.0,1655.0,1198.2,NaN,20380.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5464,Saint-Bonnet,16303,Saint-Bonnet,166.0,417.0,276.8,NaN,22860.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2713,Saint-Menges,08391,Saint-Menges,398.0,926.0,631.1,NaN,21110.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8494,Cessey,25109,Cessey,144.0,339.0,229.6,NaN,24520.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Ce dernier jeu de données contient des informations sur les voitures particulières immatriculées par commune et par type de recharge.
Cette table comporte 34926 lignes pour 57 variables.

## 3. Nettoyage et Standardisation Géographique

Le défi majeur de ce projet est la fusion des données. 

#### Choix de la variable de jointure
Les 3 jeu de données contiennent une variable indiquant le code géographique mais sous 3 noms différents :
- `code_insee_commune` pour df_irve
- `CODGEO` pour df_ve
- `Code géographique` pour df_revenus
La jointure sera faite sur ces variables.

#### Préparation des variables de jointure
Cependant les codes INSEE des communes sont souvent mal formatés.

Le diagnostic met en évidence plusieurs points d’attention :
- Présence de valeurs manquantes dans `code_insee_commune` (df_irve)  
- Présence de longueurs de codes incohérentes :
  - `code_insee_commune` (df_irve) contient des codes de longueur 3, 6 et 7
  - `CODGEO` (df_ve) contient des codes de longueur 4
- Les variables de jointure ne sont pas uniques :
  - `code_insee_commune` (df_irve)
  - `CODGEO` (df_ve)

Les clés de jointure présentent des problèmes de qualité et de format.  
Il est nécessaire de les nettoyer et les uniformiser (type, format, gestion des valeurs manquantes, unicité) avant d’effectuer la jointure.

#### Unification des codes
Nous utilisons la fonction standardize_all_codes pour garantir que chaque base dispose d'une colonne code_geo homogène sur 5 caractères.

In [5]:
dfs_to_clean = {
    "irve": (df_irve_raw, "code_insee_commune"),
    "revenu": (df_revenu_raw, "Code géographique"),
    "ve": (df_ve_raw, "CODGEO")
}

cleaned = standardize_all_codes(dfs_to_clean)
df_irve = cleaned["irve"]
df_revenu = cleaned["revenu"]
df_ve = cleaned["ve"]

Standardisation du code INSEE pour : irve (colonne : code_insee_commune)


Standardisation du code INSEE pour : revenu (colonne : Code géographique)
Standardisation du code INSEE pour : ve (colonne : CODGEO)


Une partie des données IRVE ne possède pas de code INSEE commune, mais dispose de coordonnées latitude/longitude. Plutôt que de supprimer ces lignes, nous utilisons un référentiel géographique pour retrouver la commune correspondante.
Grâce au croisement spatial avec les données de Cartiflette, nous avons pu attribuer un code commune à la quasi-totalité des points de charge.

In [6]:
# 1. Téléchargement des contours de communes via Cartiflette
communes_fr = charger_communes()

# 2. Transformation de l'IRVE en données géographiques (GeoDataFrame)
gdf_irve = creer_gdf_irve(df_irve, long_col="consolidated_longitude", lat_col="consolidated_latitude")

# 3. Jointure spatiale pour identifier la commune par les coordonnées GPS
gdf_result = joindre_communes(gdf_irve, communes_fr)

# 4. Intégration du code récupéré (code_geo_total) dans le dataframe principal
df_irve = ajouter_codes_geo(df_irve, gdf_result)

There was an error while reading the file from the URL: https://minio.lab.sspcloud.fr/projet-cartiflette/production/provider=IGN/dataset_family=ADMINEXPRESS/source=EXPRESS-COG-CARTO-TERRITOIRE/year=2022/administrative_level=COMMUNE/crs=4326/DEPARTEMENT=20/vectorfile_format=geojson/territory=metropole/simplification=0/raw.geojson
Error message: '/vsimem/pyogrio_2ea064f463ba42f5a7dd388e99e52e78' not recognized as being in a supported file format.; It might help to specify the correct driver explicitly by prefixing the file path with '<DRIVER>:', e.g. 'CSV:path'.


Nous avons maintenant 2 colonnes concernant le code géographique dans la base de données IRVE :
- `code_insee_commune` : la variable initiale
- `code_geo_total` : la variable recalculant tous les codes géographiques à partir de la latitude et la longitude

In [7]:
colonnes_irve = ["code_insee_commune", "code_geo_total"]

print("Valeurs manquantes :")
print(compter_valeurs_manquantes(df_irve, colonnes_irve))

print("\nValeurs uniques :")
print(compter_uniques(df_irve, colonnes_irve))

Valeurs manquantes :
{'code_insee_commune': np.int64(65645), 'code_geo_total': np.int64(1857)}

Valeurs uniques :
{'code_insee_commune': 9536, 'code_geo_total': 10317}


Cette manipulation permet de passer de 57919 valeurs manquantes à 1768. De plus les codes recalculés sont plus fiables que ceux initiaux. En effet, les données entrées n'étant pas toujours vérifiées, des erreurs étaient présentes. Certaines code géographiques correspondaient au code postal par exemple.

## 4. Feature Engineering : Passage à l'échelle communale

Réfléchissons aux variables à garder... Lesquelles pourraient être intéressante ?

In [8]:
list(df_irve.columns)

['nom_amenageur',
 'siren_amenageur',
 'contact_amenageur',
 'nom_operateur',
 'contact_operateur',
 'telephone_operateur',
 'nom_enseigne',
 'id_station_itinerance',
 'id_station_local',
 'nom_station',
 'implantation_station',
 'adresse_station',
 'code_insee_commune',
 'coordonneesXY',
 'nbre_pdc',
 'id_pdc_itinerance',
 'id_pdc_local',
 'puissance_nominale',
 'prise_type_ef',
 'prise_type_2',
 'prise_type_combo_ccs',
 'prise_type_chademo',
 'prise_type_autre',
 'gratuit',
 'paiement_acte',
 'paiement_cb',
 'paiement_autre',
 'tarification',
 'condition_acces',
 'reservation',
 'horaires',
 'accessibilite_pmr',
 'restriction_gabarit',
 'station_deux_roues',
 'raccordement',
 'num_pdl',
 'date_mise_en_service',
 'observations',
 'date_maj',
 'cable_t2_attache',
 'last_modified',
 'datagouv_dataset_id',
 'datagouv_resource_id',
 'datagouv_organization_or_owner',
 'created_at',
 'consolidated_longitude',
 'consolidated_latitude',
 'consolidated_code_postal',
 'consolidated_commune',
 '

In [9]:
list(df_ve.columns)

['CODGEO',
 'LIBGEO',
 'EPCI',
 'LIBEPCI',
 'DATE_ARRETE',
 'NB_VP_RECHARGEABLES_EL',
 'NB_VP_RECHARGEABLES_GAZ',
 'NB_VP',
 'code_geo']

In [10]:
list(df_revenu.columns)

['Nom géographique GMS',
 'Code géographique',
 'Libellé géographique',
 '[DISP] Nbre de ménages fiscaux',
 '[DISP] Nbre de personnes dans les ménages fiscaux',
 "[DISP] Nbre d'unités de consommation dans les ménages fiscaux",
 '[DISP] 1ᵉʳ quartile (€)',
 '[DISP] Médiane (€)',
 '[DISP] 3ᵉ quartile (€)',
 '[DISP] Écart interquartile (€)',
 '[DISP] 1ᵉʳ décile (€)',
 '[DISP] 2ᵉ décile (€)',
 '[DISP]3ᵉ décile (€)',
 '[DISP] 4ᵉ décile (€)',
 '[DISP] 6ᵉ décile (€)',
 '[DISP] 7ᵉ décile (€)',
 '[DISP] 8ᵉ décile (€)',
 '[DISP] 9ᵉ décile (€)',
 '[DISP] Rapport interdécile 9ᵉ décile/1ᵉʳ decile',
 '[DISP] S80/20',
 '[DISP] Iice de Gini',
 '[DISP] Part des revenus d’activité (%)',
 '[DISP] dont part des salaires et traitements(%)',
 '[DISP] dont part des iemnités de chômage (%)',
 '[DISP] dont part des revenus des activités non salariées (%)',
 '[DISP] Part des pensions, retraites et rentes (%)',
 '[DISP] Part des revenus du patrimoine et autres revenus (%)',
 "[DISP] Part de l'ensemble des prestat

Nos bases n'ont pas la même unité d'observation. L'IRVE liste des bornes (plusieurs par ville), alors que nous voulons une analyse par commune.

Pour pouvoir efectuer la jointure sur les codes géographiques, il est essentiel que pour chacune de nos 3 bases de données, chaque ligne corresponde à un unique code géographique.

Il faut alors réfléchir à la façon dont on veut agréger df_irve et quelles variables on souhaite garder.

Il faut également étudier la base df_ve afin de supprimer les "doublons" de code géographique.

#### IRVE
Ici, l'objectif est de mesurer l'offre de recharge par commune. Comme il peut y avoir plusieurs points de recharge par commune, il faut agréger. L'objectif final est d'expliquer ou prédire le taux de véhicules électriques local.

Après étude des nos variables (cf etude_df_irve.ipynb) nous avons sélectionner plusieurs variables nous permettant d'en créer de nouvelles, agrégées par code géographique.

Certaines variables ont été écarté directement car jugées non pertinente pour notre sujet, d'autres présetaient trop de valeurs manquantes, d'autres étaient trop peu informatives ou encore était du texte de libre donc trop difficile à utiliser avec le peu de temps dont nous disposons.

------------------------ début -------------

variables utilisées (12 variables) pour la création des variables agrégées :`code_geo_total`,`nom_operateur`,
               'implantation_station',
               'nbre_pdc',
               'puissance_nominale',
               'prise_type_ef',
               'prise_type_2',
               'prise_type_combo_ccs',
               'prise_type_chademo',
               'gratuit',
               'paiement_cb',
               'paiement_autre'

variables créées : une ligne finale = un code géographique.

|variables utilisées| Variable finale   | Construction      |
|| ----------------- | ----------------- |
|| total_pdc         | somme             |
|| puissance_moyenne | moyenne           |
|| puissance_max     | max               |
|| nb_operateurs     | nunique           |
|| top_operateur     | mode              |
|| pct_type_2        | moyenne booléenne |
|| pct_gratuit       | moyenne booléenne |
|| part_voirie       | dummies + moyenne |

------------------------ fin -------------

Commençons par rendre les données des variables sélectionnées propres avant de faire l'agrégation.

In [11]:
# Traitement IRVE : Transformation des types et agrégation
df_irve_clean = clean_irve_variables_finales(df_irve)
df_irve_final = creer_features_irve(df_irve_clean)
df_irve_final.sample(5)

,code_geo_total,total_pdc,puissance_moyenne,puissance_max,nb_operateurs,pct_type_2,pct_combo_ccs,pct_chademo,pct_type_ef,pct_gratuit,pct_paiement_cb,pct_paiement_autre,top_operateur,Parking priv rserv  la clientle,Parking priv  usage public,Parking privé réservé à la clientèle,Parking privé à usage public,Parking public,Station dédiée à la recharge rapide,Voirie
2479,26165,348,141.095238,300.0,3,0.428571,0.714286,0.047619,0.190476,0.0,0.952381,0.933333,Allego,0.0,0.0,0.0,0.309524,0.0,0.547619,0.142857
6201,59343,4012,126.611408,400.0,17,0.580986,0.535211,0.03169,0.169014,0.0,0.469751,0.584906,DRIVECO Partner Network,0.0,0.0,0.0,0.112676,0.59507,0.239437,0.052817
836,10199,4,18.000000,18.0,1,1.0,0.0,0.0,1.0,0.0,0.0,1.0,Syndicat Départemental Énergie Aube (SDEA) | F...,0.0,0.0,0.0,0.0,1.0,0.0,0.0
5914,57568,4,23.000000,24.0,1,0.5,0.5,0.0,0.0,0.0,0.0,1.0,Freshmile | FR*FR1,0.0,0.0,0.0,0.0,1.0,0.0,0.0
7762,72090,41,32.210909,50.0,2,0.636364,0.363636,0.363636,0.181818,0.0,0.0,1.0,DRIVECO,0.0,0.0,0.0,0.272727,0.727273,0.0,0.0


#### Véhicules
La base véhicules contient plusieurs dates d'observation pour une même commune.

Afin d'être cohérent avec les autres sources de données (IRVE, INSEE), nous souhaitons disposer d'une seule ligne par commune correspondant à la situation la plus récente disponible.

Nous conservons donc, pour chaque code commune (`CODGEO`), la dernière observation temporelle.

In [13]:
# Traitement VE : Sélection de la dernière observation
df_ve_final = garder_derniere_observation_commune(df_ve)
df_ve_final.head()

,CODGEO,LIBGEO,EPCI,LIBEPCI,DATE_ARRETE,NB_VP_RECHARGEABLES_EL,NB_VP_RECHARGEABLES_GAZ,NB_VP,code_geo
216883,01001,L'ABERGEMENT-CLÉMENCIAT,200069193,CC de la Dombes,2025-09-30,30,0,968,01001
432383,01002,L'ABERGEMENT-DE-VAREY,240100883,CC de la Plaine de l'Ain,2025-09-30,5,0,286,01002
541297,01004,AMBÉRIEU-EN-BUGEY,240100883,CC de la Plaine de l'Ain,2025-09-30,518,1,16080,01004
109071,01005,AMBÉRIEUX-EN-DOMBES,200042497,CC Dombes Saône Vallée,2025-09-30,68,0,2306,01005
216903,01006,AMBLÉON,200040350,CC Bugey Sud,2025-09-30,4,0,169,01006


------------------------ début -------------

A partir de cette base de données nous allons créer notre variable cible : le taux de véhicules électriques.
Cette variables est un ratio créé à aprtir des variables :
- `NB_VP` : nombre total de voitures particulières
- `NB_VP_RECHARGEABLES_EL` : nombre de voitures rechargeables / électriques.


\[
taux\_equipement\_ve = \frac{NB\_VP\_RECHARGEABLES\_EL}{NB\_VP}
\]

Ce ratio est plus pertinent que le nombre brut de véhicules électriquescar il neutralise l'effet de taille des communes.

In [14]:
df_ve_final = creer_taux_equipement_ve(df_ve_final)
df_ve_final["taux_equipement_ve"].describe().round(4)

count    35197.0000
mean         0.0229
std          0.0182
min          0.0000
25%          0.0128
50%          0.0205
75%          0.0302
max          1.0000
Name: taux_equipement_ve, dtype: float64

Nous sélectionnons pour la modélisation les variables suivantes :
- CODGEO
- NB_VP
- NB_VP_RECHARGEABLES_EL
- taux_equipement_ve

In [15]:
vars_finales = [
        "CODGEO",
        "NB_VP",
        "NB_VP_RECHARGEABLES_EL",
        "taux_equipement_ve"
    ]
df_ve_final = df_ve_final[vars_finales]

#### Revenus

Nous sélectionons pour la modélisation les variables suivantes :
- Code géographique
- [DISP] Médiane (€)
- [DISP] Iice de Gini
- [DISP] Nbre de ménages fiscaux
- [DISP] Nbre de personnes dans les ménages fiscaux
- [DISP] Part des revenus d’activité (%)
- [DISP] dont part des revenus des activités non salariées (%)
- [DISP] Part des revenus du patrimoine et autres revenus (%)

Pour plus de détails, veuillez vous référer au notebook 'etude_df_revenu'.

In [16]:
var_selec = ['Code géographique',
'[DISP] Médiane (€)',
 '[DISP] Iice de Gini',
 #garder uniquement l'un des deux
 '[DISP] Nbre de ménages fiscaux',
 '[DISP] Nbre de personnes dans les ménages fiscaux',
 #zone dynamique économiquement (potentiellement plus jeune aussi)
 '[DISP] Part des revenus d’activité (%)',
 #plus riche
 '[DISP] dont part des revenus des activités non salariées (%)',
 '[DISP] Part des revenus du patrimoine et autres revenus (%)'
]

df_revenu_final=df_revenu[var_selec]

------------------------ fin -------------

## 5. Jointure

------------------------ début -------------

Pour la base de données jointe nous avons choisis de garder les variables suivantes :
- IRVE
- Véhicules
- Revenus

------------------------ fin -------------

In [ ]:
# Fusion des trois bases
df_final = (
    df_ve_final
    .merge(df_irve_final, left_on="CODGEO", right_on="code_geo_total", how="left")
    .merge(df_revenu_final, left_on="CODGEO", right_on="Code géographique", how="left")
)

print("Shape base finale :", df_final.shape)
df_final.sample(5)

## 6. Préparation de la base de données

------------------------ début -------------

In [ ]:
# Nombre de valeurs manquantes par variable
df_final.isna().sum()

Cette nouvelle base de données comporte 35197 communes.
Parmis ces communes 24919 ne contiennet pas de données provenant de df_irve.
Cela représente environ 70% de nos communes. Après étude de la réalité, nous considéront que ces 24919 sont sans point de charge. En effet ces chiffres sont crédible car une grande partie des communes françaises ont :
- peu d’habitants
- peu de commerces
- peu de parkings publics
- peu de flux routiers
donc l'absence de borne est très fréquente.

Les IRVE sont souvent concentrées dans :
villes
axes routiers
zones commerciales
parkings publics
intercommunalités centrales
Donc beaucoup de petites communes voisines n’en ont aucune.

Attention :
Une petite commune peut ne pas avoir de borne mais être à 5 km d’une commune équipée.
Donc absence locale ne signifie pas absence d’accès réel.

In [ ]:
df_imputation = df_final.copy()

# Liste des colonnes numériques de df_irve (comptage, puissance, pourcentages)
cols_to_zero = [
    'total_pdc', 
    'puissance_moyenne', 
    'puissance_max', 
    'nb_operateurs', 
    'pct_type_2', 
    'pct_combo_ccs', 
    'pct_chademo', 
    'pct_type_ef', 
    'pct_gratuit', 
    'pct_paiement_cb', 
    'pct_paiement_autre',
    'Parking priv\x8e r\x8eserv\x8e \x88 la client\x8fle', 
    'Parking priv\x8e \x88 usage public', 
    'Parking privé réservé à la clientèle', 
    'Parking privé à usage public', 
    'Parking public', 
    'Station dédiée à la recharge rapide', 
    'Voirie'
]

# 1. Remplacement par 0 pour les chiffres
df_imputation[cols_to_zero] = df_imputation[cols_to_zero].fillna(0)

# 2. Remplacement par un label pour le texte
df_imputation['top_operateur'] = df_imputation['top_operateur'].fillna('Aucun')

df_imputation.sample(5)

In [ ]:
missing = pd.DataFrame({
    "nb_manquants": df_imputation.isna().sum(),
    "pct_manquants": (df_imputation.isna().mean() * 100).round(2)
})

missing.sort_values("nb_manquants", ascending=False)

[DISP] Iice de Gini
[DISP] dont part des revenus des activités non salariées (%)
[DISP] Part des revenus du patrimoine et autres revenus (%)
[DISP] Part des revenus d’activité (%)

Il va falloir enlever ces variables... trop de valeurs manquantes !

------------------------ fin -------------

## 7. Analyse des variables

Après le choix des variables : 
0. variable cible = taux de VE
1. faire des analyses descriptives : 
- automatiser avec une fonction
- matrice de corrélation (heatmap)
2. modélisation :
simple regression ou plus avace (random forest...)

## 8. Modélisation

Nous cherchons à prédire la variable taux_equipement_ve.

Approche 1 : Régression Lasso
Le modèle Lasso est idéal pour l'interprétabilité. Il permet de voir quelles variables "pèsent" réellement dans la décision d'achat d'un véhicule électrique.

Approche 2 : XGBoost
Ce modèle non-linéaire permet ...

Nos variables pour la modélisation sont les suivantes :

attention LASSO ne prend pas de valeurs manquantes !!

In [ ]:
features = ['total_pdc', 'puissance_max', 'nb_operateurs', 'pct_gratuit']
target = 'taux_equipement_ve' 

df_model = df_final[features + [target]].dropna()

# 2. Exécuter le Lasso (Baseline & Sélection)
model_l, metrics_l, coeffs_l = modelisation_lasso(df_model, features, target)
print(f"R2 Lasso : {metrics_l['r2']:.3f}")
print("Déterminants (Lasso) :\n", coeffs_l[coeffs_l != 0])

# 3. Exécuter le Gradient Boosting (Performance)
model_x, metrics_x, importance_x = modelisation_xgboost(df_final, features, target)
print(f"R2 XGBoost : {metrics_x['r2']:.3f}")
importance_x.plot(kind='barh', title="Importance des variables - XGBoost")

In [ ]:
features = ['[DISP] Médiane (€)', '[DISP] Iice de Gini', 'total_pdc', 'puissance_moyenne']
target = 'taux_equipement_ve'

model_lasso, metrics_lasso, coeffs = modelisation_lasso(df_master, features, target)
model_xgb, metrics_xgb, importance = modelisation_xgboost(df_master, features, target)

## 9. Analyse des résultats